<table>
    <tr>
      <td>Tratamiento y Gestión de Datos Masivos - Máster en Inteligencia Artificial - Facultad de Informática - UCM
      </td>
      <td>
      <img src="https://biblioteca.ucm.es/data/cont/media/www/pag-88746//escudo.jpg"  width=50/>
      </td>
     </tr>
</table>

# Práctica: ETL de una Red Social con PySpark
## Pablo C. Cañizares

En esta práctica construirás un pipeline ETL completo para procesar datos de **SocialLab**, una red social tipo Twitter. Partirás de 4 ficheros JSON con datos **intencionalmente sucios** y los transformarás en DataFrames limpios y agregados, siguiendo la arquitectura **Raw → Silver → Gold**.

Los datos presentan problemas reales de calidad: timestamps en 5 formatos distintos, encoding UTF-8 roto, registros duplicados, referencias huérfanas, hashtags inconsistentes y cuentas de spam.

> **Nota:** Este notebook está diseñado para ejecutarse en **Databricks**. La variable `spark` (SparkSession) ya está disponible automáticamente. Sube los 4 ficheros JSON a DBFS antes de empezar.

## Dataset: SocialLab

SocialLab es una red social tipo Twitter con 4 entidades principales:

| Fichero | Descripción | Registros aprox. | Campos principales |
|---|---|---|---|
| `users.json` | Usuarios registrados | ~3.200 | `_id`, `username`, `display_name`, `email`, `bio`, `topic`, `is_spam`, `created_at` |
| `posts.json` | Publicaciones (tweets) | ~80.000 | `_id`, `user_id`, `text`, `hashtags`, `created_at`, `likes_count` |
| `likes.json` | Likes a posts | ~200.000 | `_id`, `user_id`, `post_id`, `created_at` |
| `follows.json` | Relaciones de seguimiento | ~50.000 | `_id`, `follower_id`, `following_id`, `created_at` |

### Problemas de calidad (intencionados)

| Problema | Afecta a | Ejemplo |
|---|---|---|
| 5 formatos de timestamp | Todos | Epoch, ISO, US (`03/29/2026 06:30 PM`), EU (`29-03-2026 18:30`), Relativo (`hace 3 días`) |
| Encoding UTF-8 roto | users, posts | `Ã¡` en vez de `á` |
| Usuarios duplicados | users | Mismo email, distinto `_id` |
| Posts/likes/follows huérfanos | posts, likes, follows | Referencias a usuarios eliminados (`u_ghost_*`) |
| Self-follows | follows | `follower_id == following_id` |
| Likes duplicados | likes | Mismo usuario da like al mismo post 2 veces |
| Hashtags inconsistentes | posts | `#Cloud`, `cloud`, ` #CLOUD.` → deberían ser lo mismo |
| Spam burst | posts | 100 posts idénticos del mismo usuario |
| Campos faltantes | todos | ~4% registros sin campos obligatorios |

> **Instrucciones:** Sube los ficheros `users.json`, `posts.json`, `likes.json` y `follows.json` a Databricks (menú lateral → "Data" → "Upload to DBFS") y ajusta las rutas en la celda de Setup.

## Índice

### Bloque 0: Setup y carga
- [Ejercicio 1: Carga de los JSON](#ejercicio-1)
- [Ejercicio 2: Exploración del schema](#ejercicio-2)
- [Ejercicio 3: Conteo de registros](#ejercicio-3)

### Bloque 1: Diagnóstico de calidad
- [Ejercicio 4: Nulos por columna](#ejercicio-4)
- [Ejercicio 5: Usuarios duplicados](#ejercicio-5)
- [Ejercicio 6: Posts huérfanos](#ejercicio-6)
- [Ejercicio 7: Self-follows](#ejercicio-7)
- [Ejercicio 8: Likes duplicados](#ejercicio-8)
- [Ejercicio 9: Formatos de timestamp](#ejercicio-9)
- [Ejercicio 10: Encoding roto](#ejercicio-10)
- [Ejercicio 11: Hashtags inconsistentes](#ejercicio-11)

### Bloque 2: Limpieza — Capa Silver
- [Ejercicio 12: UDF fix_encoding](#ejercicio-12)
- [Ejercicio 13: UDF parse_timestamp](#ejercicio-13)
- [Ejercicio 14: Aplicar parse_timestamp](#ejercicio-14)
- [Ejercicio 15: Normalizar usernames](#ejercicio-15)
- [Ejercicio 16: UDF normalize_hashtags](#ejercicio-16)
- [Ejercicio 17: Eliminar registros incompletos](#ejercicio-17)
- [Ejercicio 18: Deduplicar usuarios](#ejercicio-18)
- [Ejercicio 19: Eliminar posts huérfanos](#ejercicio-19)
- [Ejercicio 20: Eliminar likes huérfanos](#ejercicio-20)
- [Ejercicio 21: Limpiar follows](#ejercicio-21)
- [Ejercicio 22: Deduplicar likes](#ejercicio-22)
- [Ejercicio 23: Detectar spam](#ejercicio-23)

### Bloque 3: Persistencia en Parquet
- [Ejercicio 24: Escribir Silver como Parquet](#ejercicio-24)
- [Ejercicio 25: Leer y verificar Parquet](#ejercicio-25)
- [Ejercicio 26: Comparar JSON vs Parquet](#ejercicio-26)

### Bloque 4: Agregaciones — Capa Gold
- [Ejercicio 27: Posts por usuario](#ejercicio-27)
- [Ejercicio 28: Likes dados y recibidos](#ejercicio-28)
- [Ejercicio 29: Followers y following](#ejercicio-29)
- [Ejercicio 30: Tabla user_stats unificada](#ejercicio-30)
- [Ejercicio 31: Métricas derivadas](#ejercicio-31)
- [Ejercicio 32: Ranking de posts](#ejercicio-32)
- [Ejercicio 33: Hashtag trends](#ejercicio-33)
- [Ejercicio 34: Ranking diario de hashtags](#ejercicio-34)
- [Ejercicio 35: Features de spam](#ejercicio-35)
- [Ejercicio 36: Aristas de interés compartido](#ejercicio-36)

### Bloque 5: Análisis final
- [Ejercicio 37: Top engagement](#ejercicio-37)
- [Ejercicio 38: Proporción de spam](#ejercicio-38)
- [Ejercicio 39: Top hashtags](#ejercicio-39)
- [Ejercicio 40: Análisis del grafo](#ejercicio-40)

---

## Setup

In [ ]:
# En Databricks la variable `spark` ya existe.
# Importamos las librerías necesarias.
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ArrayType, TimestampType, IntegerType
from pyspark.sql import Window
from datetime import datetime, timezone, timedelta
import re

# Rutas a los ficheros JSON en DBFS (ajustar si es necesario)
BASE_PATH = "/FileStore/tables/sociallab"
USERS_PATH = f"{BASE_PATH}/users.json"
POSTS_PATH = f"{BASE_PATH}/posts.json"
LIKES_PATH = f"{BASE_PATH}/likes.json"
FOLLOWS_PATH = f"{BASE_PATH}/follows.json"

# Rutas de salida
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

---

# Bloque 0: Setup y carga

---

<a id="ejercicio-1"></a>
### Ejercicio 1: Carga de los JSON
Carga los 4 ficheros JSON como DataFrames de Spark y muestra las primeras 5 filas de cada uno con `display()`.

In [ ]:
# Carga de los 4 ficheros JSON
users_raw = spark.read.json(USERS_PATH)
posts_raw = spark.read.json(POSTS_PATH)
likes_raw = spark.read.json(LIKES_PATH)
follows_raw = spark.read.json(FOLLOWS_PATH)

# Visualizar primeras filas
print("=== USERS ===")
display(users_raw.limit(5))

print("=== POSTS ===")
display(posts_raw.limit(5))

print("=== LIKES ===")
display(likes_raw.limit(5))

print("=== FOLLOWS ===")
display(follows_raw.limit(5))

---

<a id="ejercicio-2"></a>
### Ejercicio 2: Exploración del schema
Muestra el schema de cada DataFrame con `printSchema()`. Observa los tipos inferidos.

In [ ]:
print("=== USERS SCHEMA ===")
users_raw.printSchema()

print("=== POSTS SCHEMA ===")
posts_raw.printSchema()

print("=== LIKES SCHEMA ===")
likes_raw.printSchema()

print("=== FOLLOWS SCHEMA ===")
follows_raw.printSchema()

---

<a id="ejercicio-3"></a>
### Ejercicio 3: Conteo de registros
¿Cuántos registros y cuántas columnas tiene cada DataFrame? Muestra el resultado de forma clara.

In [ ]:
for name, df in [("users", users_raw), ("posts", posts_raw), ("likes", likes_raw), ("follows", follows_raw)]:
    print(f"{name}: {df.count()} filas, {len(df.columns)} columnas")

---

# Bloque 1: Diagnóstico de calidad

Antes de limpiar, debemos entender qué problemas tienen nuestros datos. En esta sección explorarás cada tipo de problema presente en el dataset.

---

<a id="ejercicio-4"></a>
### Ejercicio 4: Nulos por columna en users
Cuenta cuántos valores nulos hay en cada columna del DataFrame de usuarios. Usa `agg` con una lista de expresiones.

In [ ]:
# Contar nulos por columna
null_counts = users_raw.agg(*[
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in users_raw.columns
])
display(null_counts)

---

<a id="ejercicio-5"></a>
### Ejercicio 5: Usuarios duplicados por email
Detecta cuántos emails aparecen más de una vez. Muestra los 10 emails más repetidos con su frecuencia.

In [ ]:
dupes = (
    users_raw
    .filter(F.col("email").isNotNull())
    .groupBy("email")
    .agg(F.count("*").alias("veces"))
    .filter(F.col("veces") > 1)
    .orderBy(F.col("veces").desc())
)
print(f"Emails duplicados: {dupes.count()}")
display(dupes.limit(10))

---

<a id="ejercicio-6"></a>
### Ejercicio 6: Posts huérfanos
Encuentra posts cuyo `user_id` no existe en la tabla de usuarios. ¿Cuántos hay? Usa un **anti-join** (left join + filtro por nulo).

In [ ]:
orphan_posts = (
    posts_raw
    .join(
        users_raw.select(F.col("_id").alias("uid")),
        posts_raw.user_id == F.col("uid"),
        "left"
    )
    .filter(F.col("uid").isNull())
    .drop("uid")
)
print(f"Posts huérfanos: {orphan_posts.count()}")
display(orphan_posts.select("_id", "user_id", "text").limit(5))

---

<a id="ejercicio-7"></a>
### Ejercicio 7: Self-follows
Encuentra follows donde `follower_id == following_id`. ¿Cuántos hay?

In [ ]:
self_follows = follows_raw.filter(F.col("follower_id") == F.col("following_id"))
print(f"Self-follows: {self_follows.count()}")
display(self_follows.limit(5))

---

<a id="ejercicio-8"></a>
### Ejercicio 8: Likes duplicados
Detecta likes donde la combinación (`user_id`, `post_id`) aparece más de una vez. ¿Cuántos pares duplicados existen?

In [ ]:
dupe_likes = (
    likes_raw
    .groupBy("user_id", "post_id")
    .agg(F.count("*").alias("veces"))
    .filter(F.col("veces") > 1)
)
print(f"Pares (user, post) duplicados: {dupe_likes.count()}")
display(dupe_likes.orderBy(F.col("veces").desc()).limit(10))

---

<a id="ejercicio-9"></a>
### Ejercicio 9: Formatos de timestamp
El campo `created_at` de users tiene 5 formatos distintos. Usa `rlike` (regex) para clasificar cada registro según su formato y cuenta cuántos hay de cada tipo:
1. **Epoch**: solo dígitos, ≥10 caracteres (e.g. `1711741200`)
2. **ISO**: contiene `T` y `+` o `Z` (e.g. `2026-03-29T18:30:00Z`)
3. **US**: patrón `MM/DD/YYYY` (e.g. `03/29/2026 06:30 PM`)
4. **EU**: patrón `DD-MM-YYYY` (e.g. `29-03-2026 18:30`)
5. **Relativo**: empieza por `hace` (e.g. `hace 3 días`)

In [ ]:
classified = (
    users_raw
    .withColumn("formato",
        F.when(F.col("created_at").rlike(r"^\d{10,}$"), "Epoch")
        .when(F.col("created_at").contains("T"), "ISO")
        .when(F.col("created_at").rlike(r"^\d{2}/\d{2}/\d{4}"), "US")
        .when(F.col("created_at").rlike(r"^\d{2}-\d{2}-\d{4}"), "EU")
        .when(F.col("created_at").rlike(r"^hace"), "Relativo")
        .otherwise("Desconocido")
    )
)
display(classified.groupBy("formato").count().orderBy("formato"))

# Mostrar un ejemplo de cada formato
for fmt in ["Epoch", "ISO", "US", "EU", "Relativo"]:
    ejemplo = classified.filter(F.col("formato") == fmt).select("created_at").first()
    if ejemplo:
        print(f"{fmt}: {ejemplo.created_at}")

---

<a id="ejercicio-10"></a>
### Ejercicio 10: Encoding roto
Encuentra registros en users cuyo `display_name` o `bio` contiene el carácter `Ã` (señal de encoding UTF-8 doble). ¿Cuántos hay?

In [ ]:
broken_encoding = users_raw.filter(
    F.col("display_name").contains("Ã") |
    F.col("bio").contains("Ã")
)
print(f"Registros con encoding roto: {broken_encoding.count()}")
display(broken_encoding.select("username", "display_name", "bio").limit(5))

---

<a id="ejercicio-11"></a>
### Ejercicio 11: Hashtags inconsistentes
Explota el array de hashtags de posts, y muestra que el mismo concepto aparece en múltiples formas. Agrupa por la versión normalizada (lowercase, sin `#`, sin punto final) y cuenta cuántas variantes tiene cada uno.

In [ ]:
# Explotar hashtags
exploded = posts_raw.withColumn("tag_raw", F.explode(F.col("hashtags")))

# Normalizar para comparar
exploded = exploded.withColumn(
    "tag_norm",
    F.lower(F.regexp_replace(F.trim(F.col("tag_raw")), r"^#|\.+$", ""))
)

# Contar variantes por tag normalizado
variants = (
    exploded
    .groupBy("tag_norm")
    .agg(
        F.countDistinct("tag_raw").alias("variantes"),
        F.collect_set("tag_raw").alias("ejemplos"),
        F.count("*").alias("total_usos")
    )
    .filter(F.col("variantes") > 1)
    .orderBy(F.col("variantes").desc())
)
print(f"Hashtags con variantes inconsistentes: {variants.count()}")
display(variants.limit(10))

---

# Bloque 2: Limpieza — Capa Silver

En esta sección transformarás los datos sucios en DataFrames limpios. Cada ejercicio aborda un problema de calidad específico. Al final tendrás 4 DataFrames listos para persistir como Parquet.

### Concepto clave: UDFs (User Defined Functions)

Cuando las funciones built-in de Spark no son suficientes, puedes definir tus propias funciones con `@F.udf(TipoRetorno())`. La función recibe valores Python normales y devuelve el resultado transformado. Spark la aplica fila a fila de forma distribuida.

```python
@F.udf(StringType())
def mi_funcion(texto):
    if texto is None:
        return None
    return texto.upper()

df = df.withColumn("columna", mi_funcion(F.col("columna")))
```

---

<a id="ejercicio-12"></a>
### Ejercicio 12: UDF fix_encoding
Crea una UDF que repare el encoding roto reemplazando las secuencias corruptas por los caracteres correctos:
- `Ã¡` → `á`, `Ã©` → `é`, `Ã­` → `í`, `Ã³` → `ó`, `Ãº` → `ú`, `Ã±` → `ñ`

Aplícala a las columnas `display_name` y `bio` de users.

In [ ]:
@F.udf(StringType())
def fix_encoding(text):
    """Repara encoding UTF-8 doble."""
    if text is None:
        return None
    replacements = {
        "Ã¡": "á", "Ã©": "é", "Ã­": "í",
        "Ã³": "ó", "Ãº": "ú", "Ã±": "ñ",
        "Ã¼": "ü", "Ã": "Á", "Ã": "É",
    }
    for broken, fixed in replacements.items():
        text = text.replace(broken, fixed)
    return text

# Aplicar a users
users = (
    users_raw
    .withColumn("display_name", fix_encoding(F.col("display_name")))
    .withColumn("bio", fix_encoding(F.col("bio")))
    .withColumn("username", fix_encoding(F.col("username")))
)

# Verificar que ya no hay "Ã"
print("Registros con Ã después de fix:", users.filter(F.col("display_name").contains("Ã")).count())
display(users.select("username", "display_name", "bio").limit(5))

---

<a id="ejercicio-13"></a>
### Ejercicio 13: UDF parse_timestamp
Crea una UDF que parsee los 5 formatos de timestamp a `datetime` UTC:
1. **Epoch**: `"1711741200"` → `datetime.fromtimestamp()`
2. **ISO**: `"2026-03-29T18:30:00Z"` → `datetime.fromisoformat()` (reemplazar `Z` por `+00:00`)
3. **US**: `"03/29/2026 06:30 PM"` → `strptime("%m/%d/%Y %I:%M %p")`
4. **EU**: `"29-03-2026 18:30"` → `strptime("%d-%m-%Y %H:%M")`
5. **Relativo**: `"hace 3 días"` → `now() - timedelta(days=3)`

> **Pista:** Intenta cada formato en orden con bloques try/except. Si ninguno funciona, devuelve `None`.

In [ ]:
@F.udf(TimestampType())
def parse_timestamp(ts):
    """Parsea 5 formatos de timestamp a datetime UTC."""
    if ts is None:
        return None
    ts = ts.strip()

    # 1. Epoch
    if ts.isdigit() and len(ts) >= 10:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc)

    # 2. ISO
    try:
        return datetime.fromisoformat(ts.replace("Z", "+00:00"))
    except (ValueError, AttributeError):
        pass

    # 3. US format
    try:
        return datetime.strptime(ts, "%m/%d/%Y %I:%M %p").replace(tzinfo=timezone.utc)
    except ValueError:
        pass

    # 4. EU format
    try:
        return datetime.strptime(ts, "%d-%m-%Y %H:%M").replace(tzinfo=timezone.utc)
    except ValueError:
        pass

    # 5. Relativo
    match = re.match(r"hace (\d+) (días?|horas?)", ts)
    if match:
        amount = int(match.group(1))
        unit = match.group(2)
        now = datetime.now(timezone.utc)
        if "día" in unit:
            return now - timedelta(days=amount)
        elif "hora" in unit:
            return now - timedelta(hours=amount)

    return None

# Probar con ejemplos
test_df = spark.createDataFrame([
    ("1711741200",), ("2026-03-29T18:30:00Z",), ("03/29/2026 06:30 PM",),
    ("29-03-2026 18:30",), ("hace 3 días",)
], ["ts"])
display(test_df.withColumn("parsed", parse_timestamp(F.col("ts"))))

---

<a id="ejercicio-14"></a>
### Ejercicio 14: Aplicar parse_timestamp a los 4 DataFrames
Aplica la UDF `parse_timestamp` a la columna `created_at` de los 4 DataFrames. Después, verifica que no quedan timestamps nulos que antes no lo eran.

In [ ]:
users = users.withColumn("created_at", parse_timestamp(F.col("created_at")))
posts = posts_raw.withColumn("created_at", parse_timestamp(F.col("created_at")))
likes = likes_raw.withColumn("created_at", parse_timestamp(F.col("created_at")))
follows = follows_raw.withColumn("created_at", parse_timestamp(F.col("created_at")))

# Verificar: ¿cuántos timestamps quedaron nulos?
for name, df in [("users", users), ("posts", posts), ("likes", likes), ("follows", follows)]:
    total = df.count()
    nulos = df.filter(F.col("created_at").isNull()).count()
    print(f"{name}: {nulos}/{total} timestamps nulos ({100*nulos/total:.1f}%)")

---

<a id="ejercicio-15"></a>
### Ejercicio 15: Normalizar usernames
Convierte todos los usernames a minúsculas y reemplaza espacios por guiones bajos `_`.

In [ ]:
users = users.withColumn(
    "username",
    F.lower(F.regexp_replace(F.col("username"), r"\s+", "_"))
)
display(users.select("username").limit(10))

---

<a id="ejercicio-16"></a>
### Ejercicio 16: UDF normalize_hashtags
Crea una UDF que reciba un array de hashtags y devuelva un array limpio:
- Quitar espacios al inicio/final
- Quitar `#` inicial
- Quitar `.` final
- Convertir a minúsculas
- Eliminar tags con menos de 2 caracteres
- Eliminar duplicados dentro del mismo array

In [ ]:
@F.udf(ArrayType(StringType()))
def normalize_hashtags(tags):
    """Normaliza un array de hashtags."""
    if tags is None:
        return []
    cleaned = []
    for tag in tags:
        if tag is None:
            continue
        t = tag.strip().lstrip("#").rstrip(".").strip().lower()
        if t and len(t) > 1:
            cleaned.append(t)
    return list(set(cleaned))

# Aplicar a posts
posts = posts.withColumn("hashtags", normalize_hashtags(F.col("hashtags")))

# Verificar
display(
    posts.withColumn("tag", F.explode("hashtags"))
    .groupBy("tag").count()
    .orderBy(F.col("count").desc())
    .limit(10)
)

---

<a id="ejercicio-17"></a>
### Ejercicio 17: Eliminar registros incompletos
Elimina:
- **Users** sin `_id` o sin `username`
- **Posts** sin `_id`, `text`, `user_id` o `created_at`
- **Likes** sin `_id`, `user_id` o `post_id`
- **Follows** sin `_id`, `follower_id` o `following_id`

Muestra cuántos registros se eliminan de cada DataFrame.

In [ ]:
def count_and_filter(df, name, required_cols):
    before = df.count()
    for col_name in required_cols:
        df = df.filter(F.col(col_name).isNotNull())
        df = df.filter(F.col(col_name) != "")
    after = df.count()
    print(f"{name}: {before} → {after} (eliminados: {before - after})")
    return df

users = count_and_filter(users, "users", ["_id", "username"])
posts = count_and_filter(posts, "posts", ["_id", "text", "user_id", "created_at"])
likes = count_and_filter(likes, "likes", ["_id", "user_id", "post_id"])
follows = count_and_filter(follows, "follows", ["_id", "follower_id", "following_id"])

---

<a id="ejercicio-18"></a>
### Ejercicio 18: Deduplicar usuarios por email
Usa una **Window function** con `partitionBy("email")` y `orderBy("_id")` para asignar un número de fila. Quédate solo con la primera ocurrencia (row_number == 1).

In [ ]:
window = Window.partitionBy("email").orderBy("_id")

before = users.count()
users = (
    users
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)
after = users.count()
print(f"Users deduplicados: {before} → {after} (eliminados: {before - after})")

---

<a id="ejercicio-19"></a>
### Ejercicio 19: Eliminar posts huérfanos
Usa un `inner join` entre posts y los `_id` de users válidos para quedarte solo con posts de usuarios existentes.

In [ ]:
valid_users = users.select(F.col("_id").alias("_valid_uid"))

before = posts.count()
posts = (
    posts
    .join(valid_users, posts.user_id == F.col("_valid_uid"), "inner")
    .drop("_valid_uid")
)
after = posts.count()
print(f"Posts: {before} → {after} (huérfanos eliminados: {before - after})")

---

<a id="ejercicio-20"></a>
### Ejercicio 20: Eliminar likes huérfanos
Elimina likes que referencien a un `user_id` o `post_id` que no existen en los DataFrames limpios. Necesitas dos joins.

In [ ]:
valid_users = users.select(F.col("_id").alias("_valid_uid"))
valid_posts = posts.select(F.col("_id").alias("_valid_pid"))

before = likes.count()
likes = (
    likes
    .join(valid_users, likes.user_id == F.col("_valid_uid"), "inner")
    .drop("_valid_uid")
    .join(valid_posts, likes.post_id == F.col("_valid_pid"), "inner")
    .drop("_valid_pid")
)
after = likes.count()
print(f"Likes: {before} → {after} (huérfanos eliminados: {before - after})")

---

<a id="ejercicio-21"></a>
### Ejercicio 21: Limpiar follows
Elimina:
1. **Self-follows**: donde `follower_id == following_id`
2. **Ghost follows**: donde `follower_id` o `following_id` no existen como usuarios válidos

Muestra cuántos se eliminan en cada paso.

In [ ]:
valid_ids = users.select(F.col("_id").alias("_vid"))

before = follows.count()

# 1. Eliminar self-follows
follows = follows.filter(F.col("follower_id") != F.col("following_id"))
after_self = follows.count()
print(f"Self-follows eliminados: {before - after_self}")

# 2. Eliminar ghost follows (ambos deben existir)
follows = (
    follows
    .join(valid_ids.alias("v1"), follows.follower_id == F.col("v1._vid"), "inner")
    .drop(F.col("v1._vid"))
    .join(valid_ids.alias("v2"), follows.following_id == F.col("v2._vid"), "inner")
    .drop(F.col("v2._vid"))
)
after_ghost = follows.count()
print(f"Ghost follows eliminados: {after_self - after_ghost}")
print(f"Follows total: {before} → {after_ghost}")

---

<a id="ejercicio-22"></a>
### Ejercicio 22: Deduplicar likes
Elimina likes duplicados (mismo `user_id` + `post_id`), quedándote con el más antiguo (primer `created_at`). Usa una Window function.

In [ ]:
window = Window.partitionBy("user_id", "post_id").orderBy("created_at")

before = likes.count()
likes = (
    likes
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)
after = likes.count()
print(f"Likes deduplicados: {before} → {after} (eliminados: {before - after})")

---

<a id="ejercicio-23"></a>
### Ejercicio 23: Detectar spam
Detecta posts de spam: si un usuario publica el **mismo texto más de 3 veces**, marca esos posts como `is_spam = True`. Además, propaga el flag `is_spam` de la tabla de users a los posts de ese usuario.

> **Pista:** Usa una Window con `partitionBy("user_id", "text")` y `count("*")` para contar repeticiones.

In [ ]:
# Detectar spam por texto repetido
text_window = Window.partitionBy("user_id", "text")
posts = (
    posts
    .withColumn("_text_count", F.count("*").over(text_window))
    .withColumn(
        "is_spam",
        F.when(F.col("_text_count") > 3, True).otherwise(False)
    )
    .drop("_text_count")
)

# Marcar spam en users (por bio)
users = users.withColumn(
    "is_spam",
    F.when(F.col("is_spam") == True, True)
    .when(F.upper(F.col("bio")).contains("FOLLOW ME"), True)
    .when(F.upper(F.col("bio")).contains("BEST DEALS"), True)
    .otherwise(False)
)

# Resumen
spam_users = users.filter(F.col("is_spam") == True).count()
spam_posts = posts.filter(F.col("is_spam") == True).count()
print(f"Usuarios spam: {spam_users}")
print(f"Posts spam: {spam_posts}")

# Reset likes_count (se recalculará en gold)
posts = posts.withColumn("likes_count", F.lit(0))

---

# Bloque 3: Persistencia en Parquet

### Concepto clave: Formato columnar

Parquet es un formato **columnar** optimizado para consultas analíticas. A diferencia de JSON (orientado a filas), Parquet almacena cada columna de forma contigua, permitiendo:
- **Compresión** mucho más eficiente (datos similares juntos)
- **Lectura selectiva**: si solo necesitas 3 columnas de 20, Spark solo lee esas 3
- **Preservación del schema**: los tipos de datos se mantienen (no hay que inferirlos cada vez)

---

<a id="ejercicio-24"></a>
### Ejercicio 24: Escribir Silver como Parquet
Escribe los 4 DataFrames limpios como ficheros Parquet en la carpeta `silver/`.

In [ ]:
users.write.parquet(f"{SILVER_PATH}/users", mode="overwrite")
posts.write.parquet(f"{SILVER_PATH}/posts", mode="overwrite")
likes.write.parquet(f"{SILVER_PATH}/likes", mode="overwrite")
follows.write.parquet(f"{SILVER_PATH}/follows", mode="overwrite")

print("Silver layer escrita correctamente en Parquet.")

---

<a id="ejercicio-25"></a>
### Ejercicio 25: Leer y verificar Parquet
Relee los Parquet y verifica que el schema y el número de filas coinciden con los DataFrames originales.

In [ ]:
for name in ["users", "posts", "likes", "follows"]:
    df = spark.read.parquet(f"{SILVER_PATH}/{name}")
    print(f"{name}: {df.count()} filas, columnas: {df.columns}")
    df.printSchema()

---

<a id="ejercicio-26"></a>
### Ejercicio 26: Comparar rendimiento JSON vs Parquet
Mide el tiempo de lectura + `count()` para JSON y Parquet. ¿Cuántas veces más rápido es Parquet?

In [ ]:
import time

# JSON
start = time.time()
spark.read.json(USERS_PATH).count()
json_time = time.time() - start

# Parquet
start = time.time()
spark.read.parquet(f"{SILVER_PATH}/users").count()
parquet_time = time.time() - start

print(f"JSON:    {json_time:.3f}s")
print(f"Parquet: {parquet_time:.3f}s")
print(f"Parquet es {json_time/parquet_time:.1f}x más rápido")

---

# Bloque 4: Agregaciones — Capa Gold

La capa Gold contiene tablas analíticas derivadas de los datos limpios (Silver). Aquí construirás métricas de usuario, rankings de posts, tendencias de hashtags y features para detección de spam.

> **Nota:** A partir de aquí trabajamos con los DataFrames leídos desde Parquet (Silver). Si los tienes en memoria de los ejercicios anteriores, puedes seguir usándolos directamente.

In [ ]:
# Relectura desde Silver (por si se ha reiniciado el cluster)
users = spark.read.parquet(f"{SILVER_PATH}/users")
posts = spark.read.parquet(f"{SILVER_PATH}/posts")
likes = spark.read.parquet(f"{SILVER_PATH}/likes")
follows = spark.read.parquet(f"{SILVER_PATH}/follows")

---

<a id="ejercicio-27"></a>
### Ejercicio 27: Posts por usuario
Cuenta cuántos posts ha publicado cada usuario. Calcula también la fecha de su primer y último post.

In [ ]:
posts_count = posts.groupBy("user_id").agg(
    F.count("*").alias("posts_count"),
    F.min("created_at").alias("first_post_at"),
    F.max("created_at").alias("last_post_at"),
)
display(posts_count.orderBy(F.col("posts_count").desc()).limit(10))

---

<a id="ejercicio-28"></a>
### Ejercicio 28: Likes dados y recibidos por usuario
Calcula dos métricas:
- **Likes dados**: cuántos likes ha dado cada usuario (`likes.user_id`)
- **Likes recibidos**: cuántos likes han recibido los posts de cada usuario (requiere un join entre likes y posts para obtener el `user_id` del autor)

In [ ]:
# Likes dados
likes_given = likes.groupBy("user_id").agg(
    F.count("*").alias("likes_given")
)

# Likes recibidos (a través de los posts del usuario)
post_user = posts.select(F.col("_id").alias("post_id_ref"), "user_id")
likes_received = (
    likes
    .join(post_user, likes.post_id == F.col("post_id_ref"), "inner")
    .groupBy(post_user.user_id.alias("author_id"))
    .agg(F.count("*").alias("likes_received"))
)

display(likes_given.orderBy(F.col("likes_given").desc()).limit(5))
display(likes_received.orderBy(F.col("likes_received").desc()).limit(5))

---

<a id="ejercicio-29"></a>
### Ejercicio 29: Followers y following por usuario
Cuenta cuántos seguidores (`followers_count`) y cuántos seguidos (`following_count`) tiene cada usuario.

In [ ]:
followers_count = follows.groupBy("following_id").agg(
    F.count("*").alias("followers_count")
)
following_count = follows.groupBy("follower_id").agg(
    F.count("*").alias("following_count")
)

display(followers_count.orderBy(F.col("followers_count").desc()).limit(5))
display(following_count.orderBy(F.col("following_count").desc()).limit(5))

---

<a id="ejercicio-30"></a>
### Ejercicio 30: Tabla user_stats unificada
Une todos los contadores anteriores en una sola tabla `user_stats` usando **left joins** sobre la tabla de users. Rellena los nulos con 0 usando `fillna`.

> **Pista:** La tabla base es `users.select("_id", "username", "display_name", "is_spam", "created_at")`. Haz left join con cada tabla de contadores.

In [ ]:
user_stats = (
    users.select("_id", "username", "display_name", "is_spam", "created_at")
    .join(posts_count, users._id == posts_count.user_id, "left")
    .drop(posts_count.user_id)
    .join(likes_given, users._id == likes_given.user_id, "left")
    .drop(likes_given.user_id)
    .join(likes_received, users._id == likes_received.author_id, "left")
    .drop("author_id")
    .join(followers_count, users._id == followers_count.following_id, "left")
    .drop("following_id")
    .join(following_count, users._id == following_count.follower_id, "left")
    .drop("follower_id")
    .fillna(0, subset=["posts_count", "likes_given", "likes_received",
                        "followers_count", "following_count"])
)
display(user_stats.limit(10))

---

<a id="ejercicio-31"></a>
### Ejercicio 31: Métricas derivadas
Añade a `user_stats` tres columnas calculadas:
- `avg_likes_per_post` = likes_received / posts_count (0 si no tiene posts)
- `days_active` = días entre primer y último post + 1 (0 si no tiene posts)
- `posts_per_day` = posts_count / days_active (0 si days_active es 0)

In [ ]:
user_stats = (
    user_stats
    .withColumn(
        "avg_likes_per_post",
        F.when(F.col("posts_count") > 0,
               F.col("likes_received") / F.col("posts_count"))
        .otherwise(0)
    )
    .withColumn(
        "days_active",
        F.when(F.col("first_post_at").isNotNull(),
               F.datediff(F.col("last_post_at"), F.col("first_post_at")) + 1)
        .otherwise(0)
    )
    .withColumn(
        "posts_per_day",
        F.when(F.col("days_active") > 0,
               F.col("posts_count") / F.col("days_active"))
        .otherwise(0)
    )
    .drop("first_post_at", "last_post_at")
)

display(user_stats.orderBy(F.col("avg_likes_per_post").desc()).limit(10))

---

<a id="ejercicio-32"></a>
### Ejercicio 32: Ranking de posts por engagement real
Recalcula los likes de cada post a partir de la tabla de likes (ignorando el campo `likes_count` sucio del raw). Crea un ranking global de posts (excluyendo spam) ordenados por likes reales.

In [ ]:
real_likes = likes.groupBy("post_id").agg(
    F.count("*").alias("real_likes_count")
)

post_rankings = (
    posts
    .filter(F.col("is_spam") == False)
    .join(real_likes, posts._id == real_likes.post_id, "left")
    .drop(real_likes.post_id)
    .fillna(0, subset=["real_likes_count"])
    .withColumn("likes_count", F.col("real_likes_count"))
    .drop("real_likes_count")
    .withColumn("rank", F.row_number().over(
        Window.orderBy(F.col("likes_count").desc(), F.col("created_at").desc())
    ))
)

display(post_rankings.select("rank", "user_id", "text", "likes_count").limit(10))

---

<a id="ejercicio-33"></a>
### Ejercicio 33: Hashtag trends
Explota los hashtags de cada post (no spam), extrae la fecha, y agrupa por `(date, hashtag)` contando posts y usuarios únicos.

In [ ]:
exploded = (
    posts
    .filter(F.col("is_spam") == False)
    .withColumn("hashtag", F.explode(F.col("hashtags")))
    .withColumn("date", F.to_date(F.col("created_at")))
)

hashtag_trends = (
    exploded
    .groupBy("date", "hashtag")
    .agg(
        F.count("*").alias("post_count"),
        F.countDistinct("user_id").alias("unique_users"),
    )
)

display(hashtag_trends.orderBy(F.col("post_count").desc()).limit(10))

---

<a id="ejercicio-34"></a>
### Ejercicio 34: Ranking diario de hashtags
Usando el resultado anterior, añade un ranking por día: para cada fecha, el hashtag con más posts es #1, el siguiente #2, etc. Usa una Window con `partitionBy("date")`.

In [ ]:
hashtag_trends = hashtag_trends.withColumn(
    "rank",
    F.row_number().over(
        Window.partitionBy("date").orderBy(F.col("post_count").desc())
    )
)

# Mostrar el top-3 de cada día
display(hashtag_trends.filter(F.col("rank") <= 3).orderBy("date", "rank").limit(20))

---

<a id="ejercicio-35"></a>
### Ejercicio 35: Features para detección de spam
Construye una tabla de features por usuario para un futuro modelo de spam. Incluye:
- `posts_count`, `posts_per_day`
- `avg_text_length` (longitud media del texto de sus posts)
- `unique_texts_ratio` (textos únicos / total posts)
- `avg_hashtags_per_post` (media de hashtags por post)
- `likes_received`, `likes_given`, `followers_count`, `following_count`
- `follow_ratio` = following / followers (si followers = 0, usa following directamente)
- `label` = `is_spam` convertido a entero (0/1)

In [ ]:
# Features de posts por usuario
post_features = posts.groupBy("user_id").agg(
    F.count("*").alias("posts_count"),
    F.avg(F.length(F.col("text"))).alias("avg_text_length"),
    F.countDistinct("text").alias("unique_texts"),
    F.avg(F.size(F.col("hashtags"))).alias("avg_hashtags_per_post"),
    F.min("created_at").alias("first_post"),
    F.max("created_at").alias("last_post"),
)

post_features = (
    post_features
    .withColumn("unique_texts_ratio",
        F.when(F.col("posts_count") > 0,
               F.col("unique_texts") / F.col("posts_count"))
        .otherwise(1.0))
    .withColumn("days_active",
        F.when(F.col("first_post").isNotNull(),
               F.greatest(F.datediff(F.col("last_post"), F.col("first_post")), F.lit(1)))
        .otherwise(1))
    .withColumn("posts_per_day", F.col("posts_count") / F.col("days_active"))
    .drop("unique_texts", "first_post", "last_post")
)

# Ensamblar todo
spam_features = (
    users.select("_id", "username", "is_spam")
    .join(post_features, users._id == post_features.user_id, "left").drop("user_id")
    .join(likes_given, users._id == likes_given.user_id, "left").drop("user_id")
    .join(likes_received, users._id == likes_received.author_id, "left").drop("author_id")
    .join(followers_count, users._id == followers_count.following_id, "left").drop("following_id")
    .join(following_count, users._id == following_count.follower_id, "left").drop("follower_id")
    .fillna(0)
    .withColumn("follow_ratio",
        F.when(F.col("followers_count") > 0,
               F.col("following_count") / F.col("followers_count"))
        .otherwise(F.col("following_count")))
    .withColumn("label", F.col("is_spam").cast("integer"))
    .drop("is_spam")
)

display(spam_features.limit(10))

# Ver diferencias entre spam y legítimos
display(
    spam_features.groupBy("label").agg(
        F.avg("posts_per_day").alias("avg_posts_per_day"),
        F.avg("unique_texts_ratio").alias("avg_unique_ratio"),
        F.avg("follow_ratio").alias("avg_follow_ratio"),
        F.avg("avg_text_length").alias("avg_text_len"),
    )
)

---

<a id="ejercicio-36"></a>
### Ejercicio 36: Aristas de interés compartido
Crea una tabla de aristas entre usuarios que comparten hashtags. Para cada par de usuarios (A, B) donde A < B (evitar duplicados), cuenta cuántos hashtags tienen en común. Esto es un **self-join** sobre la tabla de hashtags por usuario.

> **Pista:** Primero crea un DataFrame `(user_id, hashtag)` con valores distintos (explode + distinct). Luego haz self-join por hashtag con la condición `a.user_id < b.user_id`.

In [ ]:
# Hashtags por usuario (sin spam)
user_hashtags = (
    posts
    .filter(F.col("is_spam") == False)
    .withColumn("hashtag", F.explode(F.col("hashtags")))
    .select("user_id", "hashtag")
    .distinct()
)

# Self-join: pares de usuarios que comparten hashtags
shared_edges = (
    user_hashtags.alias("a")
    .join(
        user_hashtags.alias("b"),
        (F.col("a.hashtag") == F.col("b.hashtag")) &
        (F.col("a.user_id") < F.col("b.user_id"))
    )
    .groupBy(
        F.col("a.user_id").alias("source"),
        F.col("b.user_id").alias("target")
    )
    .agg(F.count("*").alias("shared_hashtags"))
)

print(f"Aristas de interés compartido: {shared_edges.count()}")
display(shared_edges.orderBy(F.col("shared_hashtags").desc()).limit(10))

---

# Bloque 5: Análisis final

Con los datos limpios y agregados, responde a estas preguntas analíticas sobre SocialLab.

---

<a id="ejercicio-37"></a>
### Ejercicio 37: Top 10 usuarios con más engagement
Muestra los 10 usuarios no-spam con mayor `avg_likes_per_post` (que tengan al menos 5 posts).

In [ ]:
top_engagement = (
    user_stats
    .filter(F.col("is_spam") == False)
    .filter(F.col("posts_count") >= 5)
    .orderBy(F.col("avg_likes_per_post").desc())
    .select("username", "posts_count", "likes_received", "avg_likes_per_post", "followers_count")
    .limit(10)
)
display(top_engagement)

---

<a id="ejercicio-38"></a>
### Ejercicio 38: Proporción de spam
¿Qué porcentaje de usuarios son spam? ¿Y qué porcentaje de posts? ¿Los usuarios spam generan más posts per cápita que los legítimos?

In [ ]:
total_users = users.count()
spam_users_count = users.filter(F.col("is_spam") == True).count()

total_posts = posts.count()
spam_posts_count = posts.filter(F.col("is_spam") == True).count()

print(f"Usuarios spam: {spam_users_count}/{total_users} ({100*spam_users_count/total_users:.1f}%)")
print(f"Posts spam: {spam_posts_count}/{total_posts} ({100*spam_posts_count/total_posts:.1f}%)")

# Posts per cápita: spam vs legítimo
display(
    user_stats
    .withColumn("tipo", F.when(F.col("is_spam") == True, "Spam").otherwise("Legítimo"))
    .groupBy("tipo")
    .agg(
        F.count("*").alias("usuarios"),
        F.sum("posts_count").alias("total_posts"),
        F.avg("posts_count").alias("avg_posts_per_user"),
    )
)

---

<a id="ejercicio-39"></a>
### Ejercicio 39: Top 5 hashtags de toda la historia
Agrupa los hashtag trends y muestra los 5 hashtags con más posts totales (suma de `post_count` en todas las fechas).

In [ ]:
top_hashtags = (
    hashtag_trends
    .groupBy("hashtag")
    .agg(
        F.sum("post_count").alias("total_posts"),
        F.sum("unique_users").alias("total_users"),
    )
    .orderBy(F.col("total_posts").desc())
    .limit(5)
)
display(top_hashtags)

---

<a id="ejercicio-40"></a>
### Ejercicio 40: Análisis del grafo social
Combina las aristas FOLLOWS (de la tabla `follows`) con las aristas de SHARED_INTEREST (ejercicio 36). ¿Cuántas aristas hay de cada tipo? ¿Cuál tiene mayor peso medio?

In [ ]:
# Follow edges
follow_edges = follows.select(
    F.col("follower_id").alias("source"),
    F.col("following_id").alias("target"),
    F.lit("FOLLOWS").alias("type"),
    F.lit(1.0).alias("weight"),
)

# Shared interest edges
interest_edges = (
    shared_edges
    .select(
        "source", "target",
        F.lit("SHARED_INTEREST").alias("type"),
        F.col("shared_hashtags").cast("double").alias("weight"),
    )
)

# Unir
all_edges = follow_edges.unionByName(interest_edges)

# Resumen
display(
    all_edges.groupBy("type").agg(
        F.count("*").alias("num_aristas"),
        F.avg("weight").alias("peso_medio"),
        F.max("weight").alias("peso_max"),
    )
)

---

## ¡Práctica completada! 🎉

Has construido un pipeline ETL completo:

1. **Raw → Silver**: Limpieza de encoding, normalización de timestamps (5 formatos), deduplicación, eliminación de huérfanos, detección de spam
2. **Silver → Gold**: Métricas de usuario, rankings de engagement, trending hashtags, features para ML, grafo social

Este es el mismo pipeline que utiliza **SocialLab** internamente con PySpark.